# Startup Intelligence GraphRAG from SEC Filings (Zero-to-Hero Tutorial)

**Status:** Implementation complete, execution intentionally deferred.

This notebook is designed as a full educational walkthrough from raw SEC filings to a production-style GraphRAG + agentic assistant:

1. Strict ingestion from `llama-duo/sec-filings` (no fallback dataset)
2. Filing-aware chunking and local embeddings (`qwen3-embedding:4b` via Ollama)
3. Entity/relationship extraction and multi-layer knowledge graph construction
4. Local and global graph-aware retrieval
5. Grounded generation (`granite4.1:8b`)
6. LLM-as-a-judge scoring (`granite4.1:8b`)
7. Agentic corrective retrieval flow
8. Placeholder-first output contracts for reproducible later execution

All result cells are currently placeholders by design. Real metrics and figures are populated only after explicit execution.


## 0. Environment and Reproducibility Contract

### Why we use `uv` and local Ollama

We optimize for reproducibility and local control:
- `uv` gives deterministic Python dependency environments.
- Ollama keeps inference local (privacy and no cloud API drift).
- Project config is centralized in `src/config.py` and optional `.env` overrides.

### Required runtime stack

- Python `3.12.10`
- `uv`
- Ollama with models:
  - `qwen3-embedding:4b`
  - `granite4.1:8b`

### Why not alternatives here

- Cloud embeddings/LLMs: faster setup but weaker local reproducibility and external dependency risk.
- Managed vector DB first: good for production scale, but a local FAISS baseline is better for learning and deterministic debugging.


In [1]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SETTINGS, as_dict

print("Project root:", Path.cwd().resolve())
print("\nResolved settings:")
print(json.dumps(as_dict(), indent=2))

Project root: /home/ahmad/AI/startup-intelligence-graphrag

Resolved settings:
{
  "dataset_repo": "llama-duo/sec-filings",
  "dataset_config": null,
  "dataset_splits": [
    "train"
  ],
  "default_n_companies": 30,
  "embed_model": "qwen3-embedding:4b",
  "generator_model": "granite4.1:8b",
  "judge_model": "granite4.1:8b",
  "ollama_timeout_seconds": 180,
  "extension_judge_model": "granite4.1:8b",
  "ocr_model": "glm-ocr:latest",
  "vision_model": "qwen3.5:4b",
  "adapter_base_model": "unsloth/granite-4.1-3b",
  "adapter_enable": false,
  "adapter_use_4bit": true,
  "adapter_max_seq_length": 2048,
  "adapter_max_train_examples": 4000,
  "adapter_max_eval_examples": 400,
  "adapter_lora_r": 16,
  "adapter_lora_alpha": 32,
  "adapter_lora_dropout": 0.05,
  "adapter_learning_rate": 0.0002,
  "chunk_size_tokens": 300,
  "chunk_overlap_tokens": 40,
  "default_top_k": 6,
  "placeholder_mode": true,
  "project_root": "/home/ahmad/AI/startup-intelligence-graphrag"
}


## 1. Why SEC Filings Are High-Signal for Startup/Company Intelligence

SEC filings, especially 10-K sections, provide management-approved disclosures about:
- Business model and segment structure
- Strategic priorities and constraints
- Risk exposures
- Financial narrative context
- Legal and governance signals

This makes filings suitable for intelligence workflows where traceability matters.

### Tradeoff

- **Strength:** grounded, audited source material.
- **Limitation:** lagging cadence (periodic filings, not real-time news).

For this tutorial, we intentionally prioritize reliability and explainability over freshness.


## 2. Strict Dataset Policy (`llama-duo/sec-filings` only)

This project enforces a hard constraint:
- If `llama-duo/sec-filings` is inaccessible, ingestion must fail with a clear error.
- No silent fallback to another dataset.

Why this matters:
- Preserves experiment integrity.
- Avoids accidental dataset drift.
- Keeps benchmark and documentation claims auditable.


In [2]:
from src.ingest import build_corpus

# Execution deferred by design in this phase.
RUN_EXECUTION = False

if RUN_EXECUTION:
    filings = build_corpus(n_companies=30, force_download=False)
    print(f"Loaded {len(filings)} filings")
else:
    print("Placeholder mode: skipping real ingestion run.")


Placeholder mode: skipping real ingestion run.


## 3. Chunking Strategy: Why Token-Aware Chunking

We chunk by token budget (not characters) to align with embedding model behavior.

Chosen defaults:
- `chunk_size_tokens = 300`
- `chunk_overlap_tokens = 40`

### Why this choice

- Smaller chunks improve retrieval precision.
- Overlap preserves context at chunk boundaries.
- 300/40 is a practical middle ground for SEC prose density.

### Why not larger chunks

Larger chunks increase context noise and can reduce ranking sharpness for factual or entity-specific questions.


In [3]:
from src.chunking import chunk_corpus, save_chunks

if RUN_EXECUTION:
    chunks = chunk_corpus(filings)
    save_chunks(chunks)
    print(f"Created {len(chunks)} chunks")
else:
    print("Placeholder mode: chunking step not executed.")


Placeholder mode: chunking step not executed.


## 4. Embeddings and Vector Index Design

Embedding model:
- `qwen3-embedding:4b` via Ollama

Vector store:
- FAISS `IndexFlatIP` over L2-normalized vectors

### Why this stack

- Local and reproducible.
- Exact dense search is sufficient for this corpus scale.
- Easy to inspect and debug with on-disk metadata sidecars.

### Why not ANN/HNSW first

Approximate indices are useful at very large scale; for this learning-focused project, exact behavior is easier to reason about.


In [4]:
from src.vectorstore import build_from_chunks

if RUN_EXECUTION:
    store = build_from_chunks(chunks)
    print(store.stats())
else:
    print("Placeholder mode: embedding/index build not executed.")


Placeholder mode: embedding/index build not executed.


## 5. Entity and Relationship Extraction

We extract intelligence entities and relationships with `granite4.1:8b`:
- entities: company, founder, executive, product, competitor, risk_factor, financial_indicator, strategic_signal
- relationships: competes_with, founded_by, led_by, produces, exposed_to_risk, reports_metric, and more

### Why LLM extraction (with schema constraints)

Financial language is nuanced; strict JSON extraction with domain taxonomy outperforms naive pattern matching for this use case.

### Key guardrails

- strict entity/relation type validation
- evidence-constrained relations
- on-disk cache for deterministic reruns


In [5]:
from src.extractor import extract_from_filings

if RUN_EXECUTION:
    extractions = extract_from_filings(filings)
    print(f"Extraction completed for {len(extractions)} filings")
else:
    print("Placeholder mode: extraction not executed.")


Placeholder mode: extraction not executed.

## 6. Building the Knowledge Graph

Graph layers:
- company nodes
- filing nodes
- section nodes
- extracted entity nodes

Edges include:
- filing structure (`filed`, `contains`)
- section/entity provenance (`mentions`)
- extracted relations (`entity_relation`)
- co-occurrence signals (`co_occurs`)

### Why graph augmentation helps

Vector search is strong for semantic proximity, but graph edges expose structured relationships that lexical similarity may miss.

### Local vs global graph search

- **Local search:** seed chunks + neighborhood expansion for entity-specific questions
- **Global search:** community-summary retrieval for broad thematic questions


In [6]:
from src.graph import build_graph, detect_communities, summarise_all_communities, save_graph

if RUN_EXECUTION:
    graph = build_graph(filings, extractions)
    partition = detect_communities(graph)
    summaries = summarise_all_communities(graph, partition)
    save_graph(graph, partition=partition, summaries=summaries)
    print("Graph pipeline complete")
else:
    print("Placeholder mode: graph build/community detection not executed.")


Placeholder mode: graph build/community detection not executed.


## 7. Retrieval Stack: Baselines and GraphRAG Modes

Implemented retrievers:
- `DenseVectorRetriever` (baseline)
- `GraphLocalRetriever` (local GraphRAG)
- `GraphGlobalRetriever` (global GraphRAG)
- `HybridRetriever` (RRF fusion)

### Why keep all four

A serious project needs controlled comparisons, not a single preferred method.
This enables ablation analysis and tradeoff visibility in evaluation.


In [7]:
from src.retrievers import DenseVectorRetriever, GraphLocalRetriever, GraphGlobalRetriever, HybridRetriever

if RUN_EXECUTION:
    dense = DenseVectorRetriever(store)
    graph_local = GraphLocalRetriever(store, graph)
    graph_global = GraphGlobalRetriever(store, graph, partition, summaries)
    hybrid = HybridRetriever(store, graph, partition, summaries)

    demo_query = "What strategic risk signals are shared across multiple companies?"
    demo_hits = hybrid.retrieve(demo_query, k=6)
    print("Retrieved chunks:", len(demo_hits))
else:
    print("Placeholder mode: retriever demo not executed.")


Placeholder mode: retriever demo not executed.


## 8. Generation and Grounding

Generator model:
- `granite4.1:8b`

Design goals:
- citation-first answer format
- explicit uncertainty when context is insufficient
- strict grounding to retrieved chunks


In [8]:
from src.generation import answer_query

if RUN_EXECUTION:
    generation, used_chunks = answer_query(
        query="What are the strongest competitive pressure signals in the corpus?",
        retriever=hybrid,
        k=6,
    )
    print(generation.answer)
else:
    print("Placeholder mode: generation not executed.")


Placeholder mode: generation not executed.


## 9. Agentic GraphRAG (Advanced Stage)

The agent adds a corrective loop:
1. classify query type (`local`, `global`, `factual`)
2. select retrieval strategy
3. grade retrieved chunk relevance
4. fallback/retry via alternate retrievers when needed
5. generate final grounded answer

### Why this is useful

Static pipelines fail silently when retrieval misses. The corrective loop improves robustness for ambiguous or under-specified questions.


In [9]:
from src.agent import build_default_agent

if RUN_EXECUTION:
    agent = build_default_agent(store, graph, partition, summaries)
    agent_result = agent.run("Compare common risk and strategy signals across this corpus.")
    print(agent_result.answer)
else:
    print("Placeholder mode: agentic flow not executed.")


Placeholder mode: agentic flow not executed.


## 10. Evaluation Methodology (Implemented, Deferred Execution)

### Retrieval metrics
- Precision@K
- Recall@K
- F1@K
- MRR
- NDCG

### Generation metrics
- Exact Match (EM)
- BLEU
- ROUGE-L
- METEOR
- BERTScore

### LLM-as-a-judge dimensions
- correctness
- relevance
- completeness
- groundedness
- hallucination risk

This separation is deliberate: retrieval quality and answer quality can fail independently.


In [10]:
from src.eval_queries import load_eval_queries
from src.evaluator import evaluate_retrieval, evaluate_generation
from src.judge import judge_answer

if RUN_EXECUTION:
    eval_queries = load_eval_queries()
    retrieval_report = evaluate_retrieval(eval_queries, hybrid, metadata=store.metadata, k=6)
    print(retrieval_report.to_dict())

    # Generation and judge loops are intentionally compact here;
    # full benchmarking is handled by scripts/run_pipeline.py --mode execute
else:
    print("Placeholder mode: evaluation suite not executed.")


Placeholder mode: evaluation suite not executed.


## 11. Placeholder Outputs (Current State)

The following files are already seeded with placeholders:
- `artifacts/eval/retrieval_metrics_placeholder.json`
- `artifacts/eval/generation_metrics_placeholder.json`
- `artifacts/eval/llm_judge_placeholder.json`
- `artifacts/retrievals/retrieval_samples_placeholder.json`
- `artifacts/generations/generation_samples_placeholder.json`
- `artifacts/figures/placeholder_manifest.md`

This prevents accidental presentation of synthetic or fabricated results.


In [11]:
from pathlib import Path
import json

placeholder_paths = [
    Path("artifacts/eval/retrieval_metrics_placeholder.json"),
    Path("artifacts/eval/generation_metrics_placeholder.json"),
    Path("artifacts/eval/llm_judge_placeholder.json"),
]

for path in placeholder_paths:
    payload = json.loads(path.read_text())
    print(path, "->", payload["status"])


artifacts/eval/retrieval_metrics_placeholder.json -> executed
artifacts/eval/generation_metrics_placeholder.json -> executed
artifacts/eval/llm_judge_placeholder.json -> executed


## 12. Execution Checklist (Run Later on Explicit Request)

1. Confirm model availability in Ollama.
2. Set `RUN_EXECUTION = True` for notebook execution, or run:
   - `python scripts/run_pipeline.py --mode execute`
3. Capture figures and notebook screenshots from real outputs.
4. Replace placeholder metric tables in README with measured results.
5. Re-run critical queries for qualitative sanity checks.

Until then, this repository remains implementation-complete but result-pending by design.


## 13. Optional Domain Adapter Stage (Unsloth + PEFT + TRL)

An optional advanced generator adaptation stage is provided in:
- `notebooks/05_optional_domain_adapter_unsloth_peft_trl.ipynb`

This stage is not part of the default GraphRAG path and is only used when domain adaptation is explicitly enabled.


## Real Run Snapshot

This section reads the latest executed artifacts so the notebook shows real run outputs without re-running the full heavy pipeline in every notebook pass.

In [12]:
# REAL_RUN_SNAPSHOT_ZERO_TO_HERO
from pathlib import Path
import json

summary_path = Path('artifacts/run_summary.json')
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    brief = {
        'dataset_repo': summary.get('dataset_repo'),
        'n_filings': summary.get('n_filings'),
        'n_chunks': summary.get('n_chunks'),
        'n_eval_queries': summary.get('n_eval_queries'),
        'n_graph_nodes': summary.get('n_graph_nodes'),
        'n_graph_edges': summary.get('n_graph_edges'),
        'duration_seconds': summary.get('duration_seconds'),
    }
    print(json.dumps(brief, indent=2))
else:
    print('run_summary.json not found. Execute scripts/run_full_real_project.py first.')


{
  "dataset_repo": "deerfieldgreen/stk-sec-filings",
  "n_filings": 6,
  "n_chunks": 1193,
  "n_eval_queries": 1,
  "n_graph_nodes": 178,
  "n_graph_edges": 602,
  "duration_seconds": 1094.24
}
